#  MyAnimeList (MAL) Adathalmaz Feltáró Elemzése
## Minőség, rajongás vagy puszta hype? – A sikeres animék anatómiája

### A projekt célkitűzése és kutatási kérdése
A közösségi média és a tartalomfogyasztó platformok korában egy-egy alkotás sikerét számos különböző mutatóval mérhetjük. Jelen elemzés központi kérdése, hogy **mást mutat-e egy anime tényleges minőségéről és megítéléséről az átlagos értékelés (`Score`), a kedvencek száma (`Favorites`) és a nézői érdeklődés (`Members`)?** 

A kutatás során nemcsak e három fő dimenzió közötti korrelációkat és a toplisták átfedéseit vizsgáljuk meg, hanem felkutatjuk a feltűnően eltérő eseteket (outliereket) is: a túlértékelt közönségkedvenceket, a rétegközönségnek szóló kritikai remekműveket, valamint a kultikus klasszikusokat. Emellett górcső alá vesszük, hogy a megfigyelt mintázatok milyen összefüggésben állnak az animék **típusával (`Type`)**, **megjelenési korával (`Aired` / `Premiered`)**, **epizódszámával (`Episodes`)** és **forrásanyagával (`Source`)**.

---

### Az adathalmazok bemutatása
Az elemzés két, egymást kiegészítő, nagyméretű MyAnimeList (MAL) adathalmazra épül:

1. **`anime.csv` (Metaadatok és aggregált statisztikák):**
   * **Méret:** 17 562 egyedi anime részletes adatlapja.
   * **Tartalom:** Általános információk (cím, típus, epizódszám, stúdió, forrásanyag, műfajok, korhatár-besorolás), közösségi aktivitási mutatók (`Members`, `Favorites`, `Watching`, `Completed`, `Dropped`, `Plan to Watch`), valamint a pontszámok részletes eloszlása (`Score`, illetve a `Score-1`-től `Score-10`-ig terjedő szavazatszámok).
2. **`animelist.csv` (Felhasználói interakciók – Big Data):**
   * **Méret:** ~109 millió sor, amely 325 772 egyedi felhasználó értékeléseit és naplózott tevékenységeit tartalmazza.
   * **Tartalom:** Felhasználói azonosító (`user_id`), anime azonosító (`anime_id`), adott pontszám (`score`), megtekintési státusz (`watching_status`) és a megnézett epizódok száma (`watched_episodes`).

---

### Az elemzés felépítése (Notebook Struktúra)
A Jupyter Notebook a későbbi prezentációs logikát követve 5 fő szakaszra bomlik:

1. **Adatelőkészítés és Technikai Optimalizálás:** Hiányzó és szöveges (`"Unknown"`) értékek kezelése, adattípusok konvertálása, megjelenési évek kinyerése, valamint a 109 millió soros `animelist.csv` memóriahatékony (downcastolt) beolvasása.
2. **A Nagy Háromszög (Minőség vs. Rajongás vs. Népszerűség):** A `Score`, `Favorites` és `Members` mutatók eloszlásának, rangkorrelációjának vizsgálata, új relatív mutatók (pl. *Kedvenc-arány*) bevezetése és az outlierek (kvadránsok) azonosítása.
3. **Háttértényezők szerepe (Típus, Kor, Epizódszám):** A folytatás-torzítás (*sequel bias*), az újdonság-torzítás (*recency bias*) és az elköteleződési hatás (*commitment effect*) statisztikai igazolása.
4. **Mélyfúrás és Egyedi Mutatók:**
   * *Kaszálási és Halogatási Index:* Mely animéket hagyják félbe a legtöbben (`Drop Rate`), és melyek ragadnak be a „megnézendő” (`Plan to Watch`) listákba?
   * *Polarizációs Index:* A legmegosztóbb animék felkutatása a szavazatok szórása (`Score-1`...`Score-10`) alapján.
<!-- 5. **Big Data Insightok (`animelist.csv`):** A híres *„3 epizódos szabály”* empirikus tesztelése a lemorzsolódási adatokon, valamint a *„veterán kritikus-effektus”* vizsgálata (szigorúbban pontoznak-e azok, akik több száz animét láttak?). -->

---
*Az alábbi blokkban a szükséges Python könyvtárak importálásával és a globális megjelenítési beállításokkal indítjuk a munkát.*

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

df_anime = pd.read_csv('data/anime.csv')
#df_animelist = pd.read_csv('data/animelist.csv')


### Segédfüggvények


In [ ]:
# Generated by Github Copilot
def duration_to_minutes(value):
    if pd.isna(value) or value == "Unknown":
        return pd.NA

    value = re.sub(r"\s+", " ", str(value)).strip()

    match = re.fullmatch(
        r"(?:(\d+(?:\.\d+)?)\s*hr\.\s*)?"
        r"(?:(\d+(?:\.\d+)?)\s*min\.\s*)?"
        r"(?:(\d+(?:\.\d+)?)\s*sec\.\s*)?"
        r"(?:per ep\.)?",
        value,
    )

    if match is None:
        raise ValueError(f"Unexpected duration format: {value!r}")

    hours, minutes, seconds = match.groups()

    return (
        float(hours or 0) * 60
        + float(minutes or 0)
        + float(seconds or 0) / 60
    )

# Generated by Github Copilot
def premiered_to_year(value):
    if pd.isna(value) or value == "Unknown":
        return pd.NA

    match = re.search(r"\d{4}", str(value).strip())


    if match is None:
        raise ValueError(f"Unexpected premiered format: {value!r}")

    return int(match.group(0))

### Adattisztítás és típuskonverzió (`anime.csv`)

A nyers adathalmazban a hiányzó értékeket szöveges `"Unknown"` karakterlánc jelöli, ami miatt a Pandas számos számszerű oszlopot szövegként (`object` típusként) olvas be. Az alábbi kódrészlet elvégzi az adatok szabványosítását és a későbbi elemzéshez szükséges új változók (feature-ök) kinyerését:

* **Hiányzó értékek és numerikus mezők kezelése:** Az `"Unknown"` értékeket szabványos `pd.NA`-ra cseréljük, majd az értékeléseket (`Score`, `Score-1`...`Score-10`), a rangsort (`Ranked`) és az epizódszámot (`Episodes`) numerikus típusúvá alakítjuk.
* **Időtartam és premier éve:** Segédfüggvények segítségével a szöveges `Duration` oszlopból kinyerjük az epizódok percekben mért hosszát (`Duration_minutes`), a szezonális `Premiered` mezőből pedig a megjelenés évét (`Premiered_year`).
* **Vetítési időszak feldolgozása:** Az `Aired` oszlop intervallumait (`" to "` mentén) szétválasztjuk kezdő (`start_date`) és befejező (`end_date`) dátumokra. Az egynapos megjelenéseknél (pl. mozifilmek, OVA-k) a kezdő és záró dátum megegyezik. A két időpont különbségéből kiszámítjuk a vetítés teljes időtartamát napokban (`length_days`).
* **Ellenőrzés:** A blokk végén egy összesítő táblázatban ellenőrizzük az oszloponkénti hiányzó értékek (`NaN`) számát és a véglegesített adattípusokat.

In [ ]:
#Data uses "Unknown" instead of NaN, so we need to replace it with NaN
df_anime.replace("Unknown", pd.NA, inplace=True)
string_numerics = ["Score", "Episodes", "Ranked", "Score-1", "Score-2", "Score-3", "Score-4", "Score-5", "Score-6", "Score-7", "Score-8", "Score-9", "Score-10"]
for col in string_numerics:
    df_anime[col] = pd.to_numeric(df_anime[col], errors='raise')

df_anime["Duration_minutes"] = (
    df_anime["Duration"]
    .apply(duration_to_minutes)
    .astype("Float64")
)

df_anime["Premiered_year"] = (
    df_anime["Premiered"]
    .apply(premiered_to_year)
    .astype("Int64")
)

#df_anime[["Start_date", "End_date", "Length_days"]] = df_anime["Aired"].apply(parse_air_date)

aired = (
    df_anime["Aired"]
    .astype("string")
    .str.strip()
    .replace(["Unknown", "?"], pd.NA)
)

dates = aired.str.split(" to ", n=1, expand=True)

# A single date is both the start and end date
start_raw = dates[0]
end_raw = dates[1].fillna(dates[0])

# If either side is "?", it becomes NaT
start_date = pd.to_datetime(start_raw, errors="coerce", format="mixed")
end_date = pd.to_datetime(end_raw, errors="coerce", format="mixed")

df_anime["Start_date"] = start_date
df_anime["End_date"] = end_date

# Automatically becomes NaN when either date is missing
df_anime["Length_days"] = (end_date - start_date).dt.days

df_anime["Favorite_ratio"] = df_anime["Favorites"] / df_anime["Members"]

df_anime["Actual_Members"] = df_anime["Watching"] + df_anime["Completed"] + df_anime["On-Hold"] + df_anime["Dropped"]

display(pd.DataFrame({
    "NaN count": df_anime.isna().sum(),
    "type": df_anime.dtypes
}))

In [ ]:
numerics = string_numerics + ["Popularity", "Members", "Favorites", "Watching", "Completed", "On-Hold", "Dropped", "Length_days", "Duration_minutes", "Premiered_year", "Favorite_ratio", "Actual_Members"]

s_nancounts = df_anime[numerics].isna().sum()

desc = df_anime[numerics].describe().T

desc["NaNCount"] = s_nancounts

desc

In [ ]:
# Csak az értékeléssel rendelkező és legalább 1000 fő által listázott animéket tartjuk meg
df_corr = df_anime.dropna(subset=['Score', 'Members', 'Favorites']).copy()
df_corr = df_corr[df_corr['Members'] >= 1000].copy()

# Relatív rajongási mutató: a nézők hány százaléka jelölte kedvencnek (%)
df_corr['Favorites_Ratio'] = (df_corr['Favorites'] / df_corr['Members']) * 100

# Elemzendő oszlopok és magyar címkéik a grafikonokhoz
cols = ['Score', 'Members', 'Favorites', 'Favorites_Ratio']
labels_heatmap = [
    'Átlagpontszám\n(Score)', 
    'Nézői elérés\n(Members)', 
    'Kedvencek száma\n(Favorites)', 
    'Rajongási arány\n(Favorites %)'
]

# Spearman-rangkorrelációs mátrix kiszámítása
spearman_matrix = df_corr[cols].corr(method='spearman')


fig, ax = plt.subplots(figsize=(8, 6), dpi=120)

# Felső háromszög maszkolása az áttekinthetőségért (opcionális, itt teljes mátrixot mutatunk)
sns.heatmap(
    spearman_matrix,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True,
    linewidths=1.5,
    cbar_kws={"label": "Spearman-féle rangkorreláció (ρ)"},
    xticklabels=labels_heatmap,
    yticklabels=labels_heatmap,
    annot_kws={"size": 12},
    ax=ax
)

ax.set_title(
    "A Nagy Háromszög: Minőség, Elérés és Rajongás Korrelációja\n(Spearman ρ, szűrve: Members ≥ 1000)", 
    fontsize=13, fontweight="bold", pad=15
)
plt.xticks(rotation=0, fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()


# A vizualizációhoz log10 transzformációt alkalmazunk a ferde eloszlású változókra,
# hogy a scatter plotokon láthatóvá váljon a valódi szerkezet.
df_plot = pd.DataFrame()
df_plot['Score (1-10)'] = df_corr['Score']
df_plot['log10(Members)'] = np.log10(df_corr['Members'])
# Hozzáadunk 1-et a Favorites oszlophoz, hogy elkerüljük a log10(0) problémát
df_plot['log10(Favorites) + 1'] = np.log10(df_corr['Favorites'] + 1)
df_plot['log10(Favorite_ratio) + 0.01'] = np.log10(df_corr['Favorites_Ratio'] + 0.01)


# PairGrid inicializálása
g = sns.PairGrid(df_plot, diag_sharey=False, corner=True)

# Alsó háromszög: Scatter plotok alacsony átlátszósággal (alpha) a sűrűség érzékeltetésére
g.map_lower(
    sns.scatterplot, 
    alpha=0.15, 
    s=15, 
    color="#1f77b4", 
    edgecolor="none"
)

# Átló: Hisztogramok és sűrűségfüggvények (KDE) az egyes változók eloszlásáról
g.map_diag(
    sns.histplot, 
    kde=True, 
    color="#2b5c8f", 
    bins=60, 
    edgecolor="white",
)

# Főcím beállítása
g.fig.subplots_adjust(top=0.92)
g.fig.suptitle(
    "Tulajdonságok Páronkénti Kapcsolata és Eloszlása (Scatter Plot Mátrix, Members ≥ 1000)", 
    fontsize=15, fontweight="bold"
)

plt.show()

In [ ]:
df_outlier = df_anime.dropna(subset=['Score', 'Members']).copy()
#df_outlier = df_outlier[df_outlier['Members'] >= 1000].copy()
df_outlier["Members_log10"] = np.log10(df_outlier["Members"])

fig, ax = plt.subplots(figsize=(8, 6), dpi=120)

sns.scatterplot(data=df_outlier, x='Score', y='Members', alpha=0.25, 
    s=15, 
    color="#1f77b4", 
    ax=ax
)
plt.title("Score vs Members - Outliers Included", fontsize=14, fontweight="bold")
plt.xlabel("Score (1-10)")
plt.show()

fig, ax = plt.subplots(figsize=(8, 6), dpi=120)

sns.scatterplot(data=df_outlier, x='Score', y='Members_log10', alpha=0.25, 
    s=15, 
    color="#1f77b4", 
    ax=ax
)
plt.title("Score vs log10(Members) - Outliers Included", fontsize=14, fontweight="bold")
plt.xlabel("Score (1-10)")
plt.ylabel("log10(Members)")
plt.show()